In [ ]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
from google.colab import userdata
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import LabelEncoder

**Setting AWS credentials**

1. Click the key icon (🔑) in the left sidebar of Colab.
2. Add two new secrets:
   - AWS_ACCESS_KEY_ID: AWS access key
   - AWS_SECRET_ACCESS_KEY: AWS secret key.
3. Ensure "Notebook access" is enabled for both secrets.

In [ ]:
#@title AWS configuration
print("--- Step 1 & 2: AWS Configuration ---")
try:
    AWS_ACCESS_KEY_ID = userdata.get('AWS_ACCESS_KEY_ID')
    AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
    os.environ['AWS_ACCESS_KEY_ID'] = AWS_ACCESS_KEY_ID
    os.environ['AWS_SECRET_ACCESS_KEY'] = AWS_SECRET_ACCESS_KEY
    print("AWS credentials configured successfully.\n")
except Exception as e:
    print(f"Could not configure AWS credentials: {e}")

--- Step 1 & 2: AWS Configuration ---
AWS credentials configured successfully.



In [ ]:

#@title Read the PCA results from S3 bucket

print("--- Step 3: Loading Data from S3 ---")
S3_PATH = "s3://p9.data/fruit_categories_pca"
df = None
filtered_df = None

try:
    df = pd.read_parquet(S3_PATH)
    print("Data loaded successfully!")
    print(f"Shape of the original DataFrame: {df.shape}")
    print(f"\nTotal distinct labels found: {df['label'].nunique()}")

    selected_categories = [
        'Pineapple 1',          # Unique texture and crown
        'Banana 1',             # Unique shape and color
        'Watermelon 1',         # Unique size, pattern, and color
        'Strawberry 1',         # Unique shape and external seeds
        'Orange 1',             # Classic round shape and color
        'Blueberry 1',          # Distinct small size and color
        'Limes 1',              # Distinct green color
        'Physalis with Husk 1', # Unique papery husk
        'Apple Braeburn 1',     # Fine-grained challenge #1
        'Apple Pink Lady 1'     # Fine-grained challenge #2
    ]

    filtered_df = df[df['label'].isin(selected_categories)].copy()
    print(f"Filtered DataFrame to show {len(selected_categories)} selected categories.")
    print(f"Shape of the filtered DataFrame: {filtered_df.shape}\n")


except Exception as e:
    print(f"An error occurred during data processing: {e}")

--- Step 3: Loading Data from S3 ---
Data loaded successfully!
Shape of the original DataFrame: (9450, 130)

Total distinct labels found: 210
Filtered DataFrame to show 10 selected categories.
Shape of the filtered DataFrame: (450, 130)



In [ ]:
filtered_df.sample(3)

,path,label,pca_0,pca_1,pca_2,pca_3,pca_4,pca_5,pca_6,pca_7,...,pca_118,pca_119,pca_120,pca_121,pca_122,pca_123,pca_124,pca_125,pca_126,pca_127
8304,s3://p9.data/Test/Banana 1/115_100.jpg,Banana 1,-25.864942,0.745126,2.329424,-9.822099,19.716397,2.034501,-6.764633,14.704405,...,0.999690,0.426577,2.331889,1.250906,1.474648,-0.004451,-1.073402,-0.527724,-5.789693,2.917493
8945,s3://p9.data/Test/Apple Braeburn 1/47_100.jpg,Apple Braeburn 1,9.634293,-7.380881,-5.814996,-4.525094,5.516620,-1.692747,6.826495,-2.102699,...,0.183744,1.772015,1.204441,1.270620,1.170886,-1.530390,0.342329,-0.838266,-3.208209,-0.173910
9198,s3://p9.data/Test/Apple Pink Lady 1/228_100.jpg,Apple Pink Lady 1,4.637800,2.599301,-7.064052,1.515289,9.577519,0.969020,9.721311,-3.410280,...,-0.339859,0.221909,3.133409,0.526610,2.012886,-1.400854,1.638128,0.434600,-2.596427,0.412855


In [ ]:
#@title t-SNE 3D for Selected Categories

# Extract features and labels from the filtered DataFrame
if 'filtered_df' not in locals() or filtered_df is None:
    print("Error: 'filtered_df' not found. Please run the data loading cell first.")
else:
    # Drop both 'path' and 'label' columns to get numeric features
    filtered_features = filtered_df.drop(['path', 'label'], axis=1).values
    filtered_labels = filtered_df['label'].values

    print("Running t-SNE for 3D visualization on selected categories...")
    tsne_3d = TSNE(n_components=3,
                   perplexity=50,
                   max_iter=1000,
                   learning_rate='auto',
                   init='pca',
                   random_state=42,
                   verbose=0)

    # Run t-SNE on the filtered features
    tsne_features_3d = tsne_3d.fit_transform(filtered_features)
    print("3D t-SNE complete.")

    # Create a DataFrame for easy plotting with Plotly
    df_tsne_3d = pd.DataFrame({
        'tsne_1': tsne_features_3d[:, 0],
        'tsne_2': tsne_features_3d[:, 1],
        'tsne_3': tsne_features_3d[:, 2],
        'label': filtered_labels # Use the labels from the filtered DataFrame
    })

    print("Generating interactive 3D t-SNE plot...")

    # Create the base figure, adding a color sequence to match Matplotlib's tab10
    fig_3d_tsne = px.scatter_3d(
        df_tsne_3d,
        x='tsne_1',
        y='tsne_2',
        z='tsne_3',
        color='label',
        title='Interactive 3D t-SNE Visualization of selected fruit features',
        labels={'tsne_1': 't-SNE Component 1', 'tsne_2': 't-SNE Component 2', 'tsne_3': 't-SNE Component 3'},
        color_discrete_sequence=px.colors.qualitative.T10
    )

    # Enhance marker aesthetics
    fig_3d_tsne.update_traces(
        marker=dict(
            size=6,
            opacity=0.8
        )
    )

    fig_3d_tsne.update_layout(
        # --- Title Styling ---
        title={
            'text': "<b>t-SNE on selected fruits (3D)</b>",
            'y':0.95,
            'x':0.5,
            'xanchor': 'center',
            'yanchor': 'top',
            'font': {'size': 15}
        },
        # --- Legend Styling ---
        legend={
            'title_text':'Category',
            'font': {'size': 10},
            'title_font': {'size': 11},
            'yanchor': "top",
            'y': 1.0,
            'xanchor': "left",
            'x': 1.01,
            'bordercolor': '#cccccc',
            'borderwidth': 1
        },
        plot_bgcolor='#f8f9fa',
        scene=dict(
            xaxis=dict(gridcolor='rgba(255, 255, 255, 0.4)', backgroundcolor='#f8f9fa', linecolor='#cccccc', linewidth=1),
            yaxis=dict(gridcolor='rgba(255, 255, 255, 0.4)', backgroundcolor='#f8f9fa', linecolor='#cccccc', linewidth=1),
            zaxis=dict(gridcolor='rgba(255, 255, 255, 0.4)', backgroundcolor='#f8f9fa', linecolor='#cccccc', linewidth=1)
        ),
        template='plotly_white'
    )

    fig_3d_tsne.show()

    print("\n=== CLUSTER ANALYSIS (3D) ===")

    # To calculate metrics, we need numerically encoded labels for the filtered data
    le = LabelEncoder()
    filtered_labels_encoded = le.fit_transform(filtered_labels)


    # Calculate cluster metrics on the 3D t-SNE results
    tsne_3d_silhouette = silhouette_score(tsne_features_3d, filtered_labels_encoded)
    tsne_3d_davies_bouldin = davies_bouldin_score(tsne_features_3d, filtered_labels_encoded)

    print(f"\n3D t-SNE Metrics (for {len(le.classes_)} categories):")
    print(f"  Silhouette Score: {tsne_3d_silhouette:.3f} (higher is better)")
    print(f"  Davies-Bouldin Index: {tsne_3d_davies_bouldin:.3f} (lower is better)")

    # --- Inter-class distances ---
    print("\n=== SEPARATION ANALYSIS (3D) ===")
    print(f"\n3D t-SNE Class Centroids:")

    # Use the filtered labels and the new label names from the encoder
    centroids_3d = {cat: tsne_features_3d[filtered_labels == cat].mean(axis=0) for cat in le.classes_}

    # Find most and least separated pairs in 3D space
    distances_3d = []
    for i, cat1 in enumerate(le.classes_):
        for cat2 in le.classes_[i+1:]:
            dist = np.linalg.norm(centroids_3d[cat1] - centroids_3d[cat2])
            distances_3d.append((dist, cat1, cat2))

    distances_3d.sort()
    print(f"  Most similar: {distances_3d[0][1]} & {distances_3d[0][2]} (dist: {distances_3d[0][0]:.2f})")
    print(f"  Most distinct: {distances_3d[-1][1]} & {distances_3d[-1][2]} (dist: {distances_3d[-1][0]:.2f})")

Running t-SNE for 3D visualization on selected categories...
3D t-SNE complete.
Generating interactive 3D t-SNE plot...



=== CLUSTER ANALYSIS (3D) ===

3D t-SNE Metrics (for 10 categories):
  Silhouette Score: 0.869 (higher is better)
  Davies-Bouldin Index: 0.255 (lower is better)

=== SEPARATION ANALYSIS (3D) ===

3D t-SNE Class Centroids:
  Most similar: Apple Braeburn 1 & Apple Pink Lady 1 (dist: 4.71)
  Most distinct: Orange 1 & Pineapple 1 (dist: 20.50)
